# ⚡ Flujos de trabajo concurrentes de agentes con Microsoft Foundry (Python)

## 📋 Tutorial avanzado de procesamiento paralelo

Este cuaderno demuestra **patrones de flujo de trabajo concurrentes** usando el Microsoft Agent Framework. Aprenderás a construir flujos de trabajo de procesamiento paralelo de alto rendimiento donde múltiples agentes de IA se ejecutan simultáneamente, mejorando drásticamente el rendimiento y permitiendo procesos empresariales sofisticados multihilo.

> **Nota de migración:** Este ejemplo anteriormente hacía referencia a GitHub Models. GitHub Models está en desuso (se retirará en julio de 2026), por lo que ahora utiliza **Microsoft Foundry** a través de `FoundryChatClient`, que se dirige a la **API Responses** de Azure OpenAI.

## 🎯 Objetivos de aprendizaje

### 🚀 **Fundamentos del procesamiento concurrente**
- **Ejecución paralela de agentes**: Ejecutar varios agentes simultáneamente para máxima eficiencia
- **Orquestación de flujos de trabajo**: Coordinar operaciones concurrentes manteniendo consistencia de datos
- **Optimización del rendimiento**: Lograr una aceleración significativa mediante procesamiento paralelo
- **Gestión de recursos**: Utilizar eficientemente los recursos del modelo de IA a través de operaciones concurrentes

### 🏗️ **Patrones avanzados de concurrencia**
- **Procesamiento Fork-Join**: Dividir trabajo entre múltiples agentes y combinar resultados
- **Paralelismo en pipeline**: Solapamiento de etapas de ejecución para rendimiento continuo
- **Balanceo de carga**: Distribuir trabajo equitativamente entre los recursos de agentes disponibles
- **Puntos de sincronización**: Coordinar agentes concurrentes en etapas críticas del flujo de trabajo

### 🏢 **Aplicaciones concurrentes empresariales**
- **Procesamiento de documentos a gran volumen**: Procesar múltiples documentos simultáneamente
- **Análisis de contenido en tiempo real**: Análisis concurrente de flujos de datos entrantes
- **Optimización de procesamiento por lotes**: Maximizar rendimiento para operaciones a gran escala
- **Análisis multimodal**: Procesamiento paralelo de diferentes tipos de contenido (texto, imágenes, datos)

## ⚙️ Requisitos y configuración

### 📦 **Dependencias requeridas**

Instala Agent Framework con capacidades de flujo de trabajo concurrente:

```bash
pip install agent-framework -U
```

### 🔑 **Configuración de Microsoft Foundry**

Inicia sesión con Azure CLI (`az login`) para que `AzureCliCredential` pueda autenticar, luego configura los detalles de tu proyecto Microsoft Foundry.

**Configuración del entorno (archivo .env):**
```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-4.1-mini
```

**Consideraciones para procesamiento concurrente:**
- **Límites de tasa**: Monitorear los límites de tasa de Azure OpenAI para solicitudes concurrentes
- **Uso de recursos**: Considerar uso de memoria y CPU con múltiples agentes concurrentes
- **Manejo de errores**: Implementar recuperación robusta de errores en operaciones paralelas

### 🏗️ **Arquitectura de flujo de trabajo concurrente**

```mermaid
graph TD
    A[Inicio del flujo de trabajo] --> B[Ejecución concurrente]
    B --> C[Grupo de agentes 1]
    B --> D[Grupo de agentes 2]
    B --> E[Grupo de agentes 3]
    C --> F[Agregación de resultados]
    D --> F
    E --> F
    F --> G[Salida final]
    
    H[Microsoft Foundry] --> C
    H --> D
    H --> E
```

**Beneficios clave:**
- **⚡ Rendimiento**: Aceleración significativa mediante ejecución paralela
- **📈 Escalabilidad**: Manejar cargas de trabajo aumentadas sin incremento proporcional del tiempo
- **🔄 Eficiencia**: Mejor utilización de recursos computacionales disponibles
- **🎯 Rendimiento (Throughput)**: Procesar más trabajo en el mismo tiempo

## 🎨 **Patrones de diseño de flujos de trabajo concurrentes**

### 🔍 **Pipeline de investigación y análisis**
```
Research Task → Parallel Research Agents → Content Synthesis → Quality Review
```

### 📊 **Flujo de trabajo de procesamiento de datos**
```
Input Data → Concurrent Processing Agents → Result Aggregation → Final Report
```

### 🎭 **Pipeline de creación de contenido**
```
Content Brief → Parallel Content Generators → Review & Merge → Final Content
```

### 🔄 **Procesamiento en múltiples etapas**
```
Input → Stage 1 (Concurrent) → Stage 2 (Concurrent) → Stage 3 (Sequential) → Output
```

## 🏢 **Beneficios de rendimiento empresariales**

### ⚡ **Optimización del rendimiento (Throughput)**
- **Ejecución paralela**: Varios agentes trabajando simultáneamente
- **Utilización de recursos**: Máxima eficiencia de la capacidad disponible del modelo de IA
- **Reducción del tiempo**: Disminución significativa del tiempo total de procesamiento
- **Arquitectura escalable**: Añadir fácilmente más agentes concurrentes según necesidad

### 🛡️ **Confiabilidad y resiliencia**
- **Tolerancia a fallos**: Fallos individuales de agentes no detienen el flujo de trabajo completo
- **Aislamiento de errores**: Los problemas en una rama concurrente no afectan a las otras
- **Degradación gradual**: El sistema sigue operando incluso con capacidad reducida de agentes
- **Mecanismos de recuperación**: Reintentos automáticos y manejo de errores para operaciones fallidas

### 📊 **Monitoreo y observabilidad**
- **Seguimiento de ejecución concurrente**: Monitorear el progreso de todas las operaciones paralelas
- **Métricas de rendimiento**: Medir aceleración y ganancias de eficiencia
- **Análisis de uso de recursos**: Optimizar la asignación de agentes concurrentes
- **Identificación de cuellos de botella**: Detectar y resolver restricciones de rendimiento

¡Construyamos flujos de trabajo de IA concurrentes de alto rendimiento! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
import os
from typing import Any

from agent_framework import (
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowViz,
    handler,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv()


In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
ResearcherAgentName = "Researcher-Agent"
ResearcherAgentInstructions = "You are my travel researcher, working with me to analyze the destination, list relevant attractions, and make detailed plans for each attraction."

In [ ]:
PlanAgentName = "Plan-Agent"
PlanAgentInstructions = "You are my travel planner, working with me to create a detailed travel plan based on the researcher's findings."

In [ ]:
research_agent = provider.as_agent(
    name=ResearcherAgentName,
    instructions=ResearcherAgentInstructions,
)

plan_agent = provider.as_agent(
    name=PlanAgentName,
    instructions=PlanAgentInstructions,
)


In [ ]:
# A passthrough executor that broadcasts the user input to every agent in parallel.
class InputDispatcher(Executor):
    """Forward the user input unchanged to all participating agents."""

    @handler
    async def forward(self, text: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(text)


dispatcher = InputDispatcher(id="dispatcher")
agents = [research_agent, plan_agent]

workflow = (
    WorkflowBuilder(
        start_executor=dispatcher,
        output_executors=agents,
    )
    .add_fan_out_edges(dispatcher, agents)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
events = await workflow.run("Plan a trip to Seattle in December")
outputs = events.get_outputs()

In [ ]:
if outputs:
    print("===== Final Aggregated Responses =====")
    # outputs is a list of AgentResponse objects, one per output executor
    # (research_agent then plan_agent), in the order given to output_executors.
    for i, response in enumerate(outputs, start=1):
        print(f"{'-' * 60}\n\n{i:02d}:\n{response.text}")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Descargo de responsabilidad**:
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automatizadas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional humana. No somos responsables de cualquier malentendido o interpretación errónea que surja del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
